In [12]:
img_count=0

In [23]:
import cv2
import os
from ultralytics import YOLO
from PIL import Image

# ---------------- CONFIG ----------------
video_path = "gowtham/seatbelt5.mp4"
output_folder = "roi_images/val/seatbelt/"
os.makedirs(output_folder, exist_ok=True)

model = YOLO("yolov8n.pt")   # Load YOLOv8 nano model

frame_skip = 6   # 30 FPS → 5 FPS

CONF_THRESHOLD = 0.3   # optional (adjust if needed)

# ---------------- VIDEO READ ----------------
cap = cv2.VideoCapture(video_path)
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    # Skip frames
    if frame_id % frame_skip != 0:
        continue

    # ---------------- YOLO DETECTION ----------------
    results = model(frame)

    print(results[0].names)  # Print detected class names for debugging

    for r in results:
        boxes = r.boxes

        if boxes is None:
            continue

        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            print(f"Detected class ID: {class_id}, Confidence: {confidence:.2f}")  # Debug print

            # ✅ FILTER ONLY PERSON (class 0)
            if class_id != 0:
                continue

            # ✅ Confidence filter
            if confidence < CONF_THRESHOLD:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # ---------------- ROI EXTRACTION ----------------
            roi = frame[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            
            # ---------------- RESIZE TO 224x224 ----------------

            roi_resized = cv2.resize(roi, (224, 224))
            # ---------------- SAVE IMAGE ----------------
            save_path = os.path.join(output_folder, f"frame_{img_count}.jpg")
            cv2.imwrite(save_path, roi_resized)

            img_count += 1

print(f"✅ Saved {img_count} PERSON ROI images to '{output_folder}'")

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 1 tie, 65.3ms
Speed: 58.3ms preprocess, 65.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'be

In [10]:
cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)
print("Video opened:", cap.isOpened())

Video opened: False
